# CMU 11-868: LLM Inference with JAX (TPU v5e)

Run this notebook in a **TPU v5e** environment. Implement each section inside `# BEGIN Part X.Y` ... `# END Part X.Y` in `model/` (see docstrings for tensor shapes and constraints), then run the test cell below each part and the full grading cell at the end.

| Part | File | Functions |
| :--- | :--- | :--- |
| **Part 1** (`1.1`–`1.3`) | `model/layers.py` | `rms_norm`, `apply_rope`, `swiglu_mlp` |
| **Part 2** (`2.1`–`2.2`) | `model/kv_cache.py` | `init_kv_cache`, `update_kv_cache` |
| **Part 3** (`3.1`–`3.2`) | `model/attention.py` | `make_attention_mask`, `gqa_attention` |
| **Part 4** (`4.1`–`4.3`) | `model/transformer.py` | `transformer_block`, `prefill`, `decode_step` |
| **Part 5** (`5.1`–`5.2`) | `model/generate.py` | `sample_next_token`, `generate` |

In [ ]:
%load_ext autoreload
%autoreload 2

import time
import jax
import jax.numpy as jnp
import numpy as np
import pytest
import model

print(f"JAX {jax.__version__} | backend={jax.default_backend()} | devices={jax.devices()}")

### JAX Primitives Reference
The cell below demonstrates the five JAX primitives used across Parts 1–5: functional slice updates (`jax.lax.dynamic_update_slice`), explicit PRNG splitting (`jax.random.split`), parameter PyTrees, single-trace static-shape `@jax.jit`, and compiled loops (`jax.lax.scan`).

Syntax reference: [JAX Key Concepts](https://docs.jax.dev/en/latest/key-concepts.html) · [JAX API Reference](https://docs.jax.dev/en/latest/jax.html) · [Thinking in JAX](https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html)

In [ ]:
# JAX primitives reference: dynamic_update_slice, PRNG split, PyTrees, static-shape JIT, lax.scan
buf = jnp.zeros((2, 6), dtype=jnp.float32)
buf_updated = jax.lax.dynamic_update_slice(buf, jnp.array([[10.0, 20.0], [30.0, 40.0]]), start_indices=(0, 2))
print("dynamic_update_slice:\n", buf_updated)

key = jax.random.PRNGKey(0)
key, k1, k2 = jax.random.split(key, 3)
print("PRNG split:", jax.random.normal(k1, (2,)), jax.random.normal(k2, (2,)))

cfg_demo = model.ModelConfig.from_json("configs/model_test.json")
params_demo = model.init_model_params(cfg_demo, key)
print("PyTree leaves:", len(jax.tree_util.tree_leaves(params_demo)))

traces = 0

@jax.jit
def write_one(arr, val, idx):
    global traces
    traces += 1
    return jax.lax.dynamic_update_slice(arr, val, (0, idx))

arr = jnp.zeros((2, 8))
for step in range(4):
    arr = write_one(arr, jnp.ones((2, 1)) * (step + 1), jnp.asarray(step, dtype=jnp.int32))
print(f"JIT trace count (4 dynamic indices): {traces}")

_, history = jax.lax.scan(lambda c, _: (c + 1, c + 1), init=jnp.array(0), xs=None, length=5)
print("lax.scan history:", history)

---
## Part 1: Core LLaMA Layers (`model/layers.py`)

### Part 1.1: `rms_norm(x, weight, eps=1e-6)`
$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}} \odot w$$

### Part 1.2: `apply_rope(x, positions, cos, sin)`
Using the **rotate-half** convention, split each head vector $x = [x^{(1)}, x^{(2)}]$ with $x^{(1)}, x^{(2)} \in \mathbb{R}^{D/2}$ and rotate by angle table $(\cos, \sin)$ indexed by token position $p$:
$$\text{RoPE}(x, p) = \big[x^{(1)} \odot \cos_p - x^{(2)} \odot \sin_p,\;\; x^{(1)} \odot \sin_p + x^{(2)} \odot \cos_p\big]$$

### Part 1.3: `swiglu_mlp(x, w_gate, w_up, w_down)`
$$\text{SwiGLU}(x) = \big(\text{SiLU}(x W_{\text{gate}}) \odot (x W_{\text{up}})\big) W_{\text{down}}$$

In [ ]:
pytest.main(["-v", "-k", "part_1", "tests/test_inference.py"])

---
## Part 2: Static-Shape KV Cache (`model/kv_cache.py`)

![Left-Padded Static KV Cache Layout](figures/kv_cache_left_pad.png)

*Left-padded static KV cache ($B = 2, P = 5, S = 8$): aligning the last real prompt token at index $P - 1$ lets every sequence write decode step $t$ to the same slot `cache_index = P + t` while tracking independent RoPE positions.*

### Part 2.1: `init_kv_cache(config, batch_size, max_seq_len, dtype=jnp.float32)`
Allocate a zero-initialized static KV cache for all layers.

### Part 2.2: `update_kv_cache(layer_cache, k_new, v_new, cache_index)`
Functionally write `k_new` and `v_new` into sequence slots `[cache_index, cache_index + T)` of a single layer's cache.

In [ ]:
pytest.main(["-v", "-k", "part_2", "tests/test_inference.py"])

---
## Part 3: Masking & Grouped-Query Attention (`model/attention.py`)

![Grouped-Query Attention](figures/gqa_grouping.png)

*Grouped-Query Attention ($H = 4, H_{\text{kv}} = 2, G = H / H_{\text{kv}} = 2$): query heads are grouped as `[B, T, H_kv, G, D]` so each KV head is shared without duplicating `k_cache` or `v_cache`.*

### Part 3.1: `make_attention_mask(cache_valid_mask, cache_index, query_len)`
Entry `(b, 0, t, s)` is `True` iff key slot `s` is valid (`cache_valid_mask[b, s]`) **and** causal ($s \le \text{cache\_index} + t$).

### Part 3.2: `gqa_attention(q, k_cache, v_cache, attn_mask)`
$$\text{Attn}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{D}} + M\right) V$$

In [ ]:
pytest.main(["-v", "-k", "part_3", "tests/test_inference.py"])

---
## Part 4: Transformer Block, Prefill, and Decode Step (`model/transformer.py`)

![Pre-Norm LLaMA Transformer Block](figures/llama_block.png)

*Pre-norm LLaMA Transformer block: input `x` (`[B, T, d_model]`) $\rightarrow$ output (`[B, T, d_model]`) and updated `LayerKVCache`.*

### Part 4.1: `transformer_block(x, layer_params, layer_cache, positions, cos, sin, cache_index, attn_mask, config)`
Pre-norm LLaMA decoder block as illustrated above (RMSNorm $\rightarrow$ QKV + RoPE $\rightarrow$ KV-cache update $\rightarrow$ GQA + $W_o$ + residual $\rightarrow$ RMSNorm $\rightarrow$ SwiGLU + residual).

### Part 4.2: `prefill(params, prompt_ids, prompt_mask, config, max_seq_len, cos, sin)`
Compute-bound prefill pass over a left-padded prompt batch, populating KV cache slots `[0, P)` and returning logits, cache, valid mask, and each sequence's next RoPE position.

### Part 4.3: `decode_step(params, token_ids, positions, cache, cache_valid_mask, cache_index, cos, sin, config)`
Single-token ($T = 1$) autoregressive decode step writing K/V to `cache_index` and returning next-token logits, updated cache, and updated valid mask.

In [ ]:
pytest.main(["-v", "-k", "part_4", "tests/test_inference.py"])

---
## Part 5: Sampling & Batched Generation (`model/generate.py`)

### Part 5.1: `sample_next_token(logits, rng_key, temperature=0.0, top_k=0)`
- **Contract**: `logits: [B, V]` $\rightarrow$ `[B]` (`int32`).
- **Semantics**: Greedy `argmax` when `temperature <= 0.0`; otherwise temperature-scaled categorical sampling, restricted to the `top_k` highest logits when `0 < top_k < V`.

### Part 5.2: `generate(params, prompt_ids, prompt_mask, config, max_new_tokens, rng_key, temperature=0.0, top_k=0)`
- **Contract**: `prompt_ids, prompt_mask: [B, P]` $\rightarrow$ `generated_ids: [B, max_new_tokens]` (`int32`, excluding prompt).
- **Constraints**:
  - Compiled with `@jax.jit` (static: `config`, `max_new_tokens`, `temperature`, `top_k`) and uses `jax.lax.scan` for autoregressive decode (no Python decode loop).
  - Must handle `max_new_tokens == 1`, split PRNG keys per step for deterministic stochastic sampling, and produce identical tokens for left-padded batched prompts vs. unpadded `B = 1` runs.

In [ ]:
pytest.main(["-v", "-k", "part_5", "tests/test_inference.py"])

---
## Full Grading Suite (TPU v5e)

Runs all correctness tests and the TPU v5e KV-cache speedup test ($\ge 100\times$ over uncached recomputation on TPU v5e). Pass `"-k", "not tpu"` if running locally on CPU.

In [ ]:
exit_code = pytest.main(["-v", "tests/test_inference.py"])
assert exit_code == 0, "Some tests failed! See output above."

---
## Batch-Size Roofline Benchmark (TPU v5e)

Measures steady-state decode latency (TPOT) and throughput (tok/s) across batch sizes `(1, 4, 16, 32)` against the analytical TPU v5e roofline.

In [ ]:
device = jax.devices()[0]
dtype = jnp.bfloat16 if device.platform in ("tpu", "gpu") else jnp.float32
dtype_bytes = 2 if dtype == jnp.bfloat16 else 4

cfg_bench = model.ModelConfig.from_json("configs/model_benchmark.json")
params_bench = model.init_model_params(cfg_bench, jax.random.PRNGKey(2026), dtype=dtype)
prompt_len, max_new = 64, 64

print(f"Device: {device} | dtype: {dtype} | prompt_len={prompt_len}, max_new={max_new}")
print(f"{'Batch':>6} | {'Total (ms)':>10} | {'TPOT (ms)':>10} | {'Tok/s':>10} | {'Roofline TPOT (ms)':>18}")
print("-" * 68)
for bsz in (1, 4, 16, 32):
    p_ids = jax.random.randint(jax.random.PRNGKey(bsz), (bsz, prompt_len), 1, cfg_bench.vocab_size, dtype=jnp.int32)
    p_mask = jnp.ones((bsz, prompt_len), dtype=jnp.bool_)

    out = model.generate(params_bench, p_ids, p_mask, cfg_bench, max_new, jax.random.PRNGKey(0), temperature=0.0)
    out.block_until_ready()

    t0 = time.perf_counter()
    out = model.generate(params_bench, p_ids, p_mask, cfg_bench, max_new, jax.random.PRNGKey(0), temperature=0.0)
    out.block_until_ready()
    elapsed_s = time.perf_counter() - t0

    tpot_ms = (elapsed_s / max_new) * 1e3
    tok_per_s = (bsz * max_new) / elapsed_s
    roof = model.estimate_tpu_v5e_roofline(cfg_bench, bsz, prompt_len + max_new // 2, dtype_bytes=dtype_bytes)
    print(f"{bsz:>6} | {elapsed_s*1e3:>10.2f} | {tpot_ms:>10.3f} | {tok_per_s:>10.1f} | {roof['roofline_tpot_ms']:>18.4f}")

---
## Optional: HuggingFace LLaMA Checkpoint Generation

In [ ]:
RUN_HF_DEMO = False

if RUN_HF_DEMO:
    model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
    dtype = jnp.bfloat16 if jax.default_backend() in ("tpu", "gpu") else jnp.float32
    cfg_hf, params_hf, tok = model.load_hf_llama_weights(model_id, dtype=dtype)

    texts = [
        "The key advantage of a static-shape KV cache on TPUs is",
        "In one sentence, explain why LLM decode is memory-bandwidth bound:",
    ]
    encoded = [tok.encode(t) for t in texts]
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    p_ids, p_mask = model.left_pad_prompts(encoded, pad_token_id=pad_id)

    t0 = time.perf_counter()
    gen_ids = model.generate(params_hf, p_ids, p_mask, cfg_hf, max_new_tokens=48, rng_key=jax.random.PRNGKey(0), temperature=0.0)
    gen_ids.block_until_ready()
    print(f"Compile + first run: {(time.perf_counter() - t0)*1e3:.1f} ms")

    t0 = time.perf_counter()
    gen_ids = model.generate(params_hf, p_ids, p_mask, cfg_hf, max_new_tokens=48, rng_key=jax.random.PRNGKey(0), temperature=0.0)
    gen_ids.block_until_ready()
    dt = time.perf_counter() - t0
    print(f"Steady-state: {dt*1e3:.1f} ms ({(len(texts)*48)/dt:.1f} tok/s)\n")

    for prompt_text, out_tokens in zip(texts, np.asarray(gen_ids)):
        print("PROMPT:", prompt_text)
        print("OUTPUT:", tok.decode(out_tokens, skip_special_tokens=True))
        print("-" * 60)
else:
    print("Set RUN_HF_DEMO = True to run generation with SmolLM2-135M.")